# Imports

In [12]:
import librosa
import pandas as pd
from typing import Callable, Union, List
import os
from pathlib import Path
import sys
from pprint import pprint

from audio_dataset import RavdessRawData
from Preprocess import audio_to_waveform, trim_silence

# Functions

In [ ]:
def extract_audio_statistics(
    audio_paths: List[Path],
    stat_func: Callable[[Path], dict]
) -> pd.DataFrame:
    """
    Generate a DataFrame with statistics for each audio file.

    Each row corresponds to an audio file.
    Columns include the file path and attributes returned by `stat_func`.

    Parameters:
    - audio_paths: List of audio file paths.
    - stat_func: A function that takes a path and returns a dict of stats.

    Returns:
    - pd.DataFrame with one row per file and one column per attribute.
    """
    records = []
    for path in audio_paths:
        stats = stat_func(path)
        stats["path"] = str(path)
        records.append(stats)
    return pd.DataFrame(records)


def no_silence_duration_stat(path: Path) -> dict:
    """
    function to extract duration of the no-silence part of an audio file. meaning the duration after trimming silence from the start and end of the audio file.
    """
    # Here you would implement the logic to get the duration of the audio file.
    waveform, sample_rate = audio_to_waveform(path)
    trimmed_waveform = trim_silence(waveform)
    duration = librosa.get_duration(y=trimmed_waveform, sr=sample_rate)
    return {"duration": duration} 

## Change Working Dir To the Project Working Dir

In [15]:
# change the dir to the grandparent directory of the current working directory

current_dir = Path(os.getcwd())
grandparent_dir = current_dir.parent.parent
os.chdir(grandparent_dir)

In [17]:
os.getcwd()  # Check the cwd has updated

'c:\\Users\\noams\\Python Projects\\Audio_processing_project'

# Data Examination

In [18]:
ravdess_raw_data = RavdessRawData()
audio_paths_with_labels = list(ravdess_raw_data.all_data)
audio_paths = [path for path, _ in audio_paths_with_labels]
ravdess_silenced_duraion = extract_audio_statistics(audio_paths, no_silence_duration_stat)
print(ravdess_silenced_duraion.head())  # Display the first few rows of the DataFrame

   duration                                               path
0     1.664  RAVDESS\original_data\Actor_18\03-01-03-02-02-...
1     1.952  RAVDESS\original_data\Actor_12\03-01-03-02-02-...
2     1.888  RAVDESS\original_data\Actor_08\03-01-07-01-02-...
3     1.344  RAVDESS\original_data\Actor_13\03-01-08-02-01-...
4     2.560  RAVDESS\original_data\Actor_03\03-01-06-02-01-...


In [21]:
# show statistics of the audio files
ravdess_silenced_duraion.describe()  # Display the statistics of the DataFrame

,duration
count,1440.000000
mean,1.732832
std,0.349538
min,0.864000
25%,1.504000
50%,1.664000
75%,1.920000
max,3.412437
